# 06 · Tier 3: GNN embedding + XGBoost (the primary commercial candidate)

One unit per dataset × graph builder × seed; needs the 05 unit with the same coordinates. The GNN's penultimate
layer (64-d) for every train/val/test row is appended to the tabular features and the Tier 0 XGBoost recipe is fit again.

This is the deployable shape: the tree model stays the decision engine, the graph enters as a feature block that can be
computed in batch. Two numbers matter: the paired gain over Tier 0 XGBoost, and how that gain compares with the
Tier 1 hand-built features (does the lift need a neural network?). We also report what share of the model's
attribution mass sits on embedding columns.

**How this notebook runs.** Never by hand in an editor. `scripts/run_nb.sh 06_hybrid dataset=... seed=...` executes it
with papermill on the GPU node, writes the executed copy to `runs/06_hybrid/<dataset>/<graph_builder>/<seed>/executed.ipynb`,
and the last cell writes a `DONE` marker. If the process dies, the wrapper writes `FAILED` and the next `run_stage.sh`
picks the unit up again; checkpoints inside the run directory mean little work is repeated.

In [ ]:
# Parameters (papermill overrides these; keep names and types)
dataset = "dgraphfin"
seed = 0
graph_builder = "native"
run_dir = ""       # empty -> runs/<stage>/<dataset>/<graph_builder>/<seed>
force = 0          # 1 -> redo work even if checkpoints exist
quick = 0          # 1 -> subsampled data, few epochs; smoke test only

In [ ]:
import json
import shutil
import time
import warnings
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from src import ckpt, config, paths, viz
from src.eval import results

STAGE = "06_hybrid"
warnings.filterwarnings("ignore")
seed = int(seed)
force, quick = bool(int(force)), bool(int(quick))
run_dir = Path(run_dir) if run_dir else paths.run_dir(STAGE, dataset, graph_builder, seed)
run_dir.mkdir(parents=True, exist_ok=True)
t0 = time.time()
print(f"stage={STAGE} dataset={dataset} graph_builder={graph_builder} seed={seed} quick={quick} force={force}")
print("run_dir:", run_dir)
pd.Series(paths.describe())

In [ ]:
from src.data import splits
from src.data.base import Bundle

cfg = config.load_config(dataset)
models_cfg = config.quick_override(cfg, quick)
bundle = Bundle.load(paths.processed_dir(dataset, quick))
split = splits.load_split(dataset, quick)
X, y = bundle.X, bundle.y.to_numpy()
tr, va, te = split["train"], split["val"], split["test"]
allowed = np.sort(np.concatenate([tr, va]))          # rows that may take part in graph construction
X_tr, X_va, X_te = X.iloc[tr], X.iloc[va], X.iloc[te]
y_tr, y_va, y_te = y[tr], y[va], y[te]
gdir = paths.graphs_dir(dataset, quick)
n_boot = 200 if quick else cfg["bootstrap"]["n_boot"]
print(bundle.summary())
print({p: len(split[p]) for p in splits.PARTS}, "| test positive rate:", round(float(np.nanmean(y_te)), 4))

## Load embeddings from stage 05

In [ ]:
from src.eval import metrics
from src.models import hybrid as HY
from src.models import xgb as XGB

gnn_dir = paths.run_dir("05_gnn", dataset, graph_builder, seed)
assert ckpt.is_done(gnn_dir), f"05_gnn unit not DONE at {gnn_dir}"
emb_index = np.load(gnn_dir / "emb_index.npy")
archs = sorted(p.stem.replace("emb_", "") for p in gnn_dir.glob("emb_*.npy") if p.stem != "emb_index")
row_of = -np.ones(bundle.n, dtype=np.int64)
row_of[emb_index] = np.arange(len(emb_index))
assert (row_of[np.concatenate([tr, va, te])] >= 0).all(), "embeddings missing for some labelled rows"
print("architectures with embeddings:", archs)

## Fit the hybrid per architecture

In [ ]:
preds, rows, shares, timings = {}, {}, {}, {}
xgb_params = dict(models_cfg["xgb"])
for arch in archs:
    E = np.load(gnn_dir / f"emb_{arch}.npy").astype(np.float32)
    XE = HY.attach_embeddings(X, E[row_of], prefix=f"emb_{arch}_")
    t = time.time()
    booster = XGB.fit_xgb(XE.iloc[tr], y_tr, XE.iloc[va], y_va, xgb_params, seed=seed)
    pv, pt = XGB.predict_xgb(booster, XE.iloc[va]), XGB.predict_xgb(booster, XE.iloc[te])
    timings[arch] = round(time.time() - t, 1)
    name = f"hybrid_{arch}"
    preds[name] = (pv, pt)
    rows[name] = results.metrics_block(y_va, pv, y_te, pt)
    shares[name] = HY.contribution_share(booster, XE.iloc[va], prefix=f"emb_{arch}_")
    XGB.save_xgb(booster, run_dir / "models" / f"{name}.json")
    print(f"{name}: val AUC {metrics.auc(y_va, pv):.4f}  test AUC {metrics.auc(y_te, pt):.4f}  "
          f"embedding attribution share {shares[name]['emb_share']:.1%}  {timings[arch]}s")

## Against Tier 0 XGBoost, the pure GNN, and Tier 1 features

In [ ]:
base = results.baseline_predictions(dataset, seed, "xgb", "test")
compare = results.compare_against(base, te, y_te, {k: v[1] for k, v in preds.items()}, n_boot=n_boot, seed=seed)

# the same comparison for the pure GNN and Tier 1 (if those units exist) so the three lifts sit on one chart
extra = {}
for arch in archs:
    g = results.load_predictions(gnn_dir, arch, "test")
    if g is not None:
        extra[f"gnn_{arch}"] = g.set_index("position").loc[te, "p"].to_numpy()
t1 = results.load_predictions(paths.run_dir("04_graph_features_xgb", dataset, graph_builder, seed), "xgb_graphfeat", "test")
if t1 is not None:
    extra["tier1_graphfeat"] = t1.set_index("position").loc[te, "p"].to_numpy()
compare_extra = results.compare_against(base, te, y_te, extra, n_boot=n_boot, seed=seed) if extra else []
cmp_df = pd.DataFrame(compare + compare_extra)
fig = viz.plot_intervals(cmp_df[cmp_df.metric == "auc"], title=f"Test AUC minus XGBoost ({dataset}/{graph_builder}, seed {seed})")
viz.savefig(fig, run_dir, "auc_vs_xgb")
plt.show()
cmp_df[["model", "metric", "base", "new", "diff", "ci_low", "ci_high", "significant"]].round(4)

## Where the hybrid looks

In [ ]:
share_df = pd.DataFrame({k: {"emb_share": v["emb_share"], "tab_share": v["tab_share"]} for k, v in shares.items()}).T
fig, ax = plt.subplots(figsize=(5, 2.8))
share_df.plot.barh(stacked=True, ax=ax); ax.set(title="Attribution mass: embedding vs tabular columns", xlim=(0, 1))
viz.savefig(fig, run_dir, "attribution_share")
plt.show()
_, p_base, _ = results.align(base, pd.DataFrame({"position": te, "y": y_te, "p": list(preds.values())[0][1]}))
fig = viz.plot_roc_pr(y_te, {"xgb (tier 0)": p_base, **{k: v[1] for k, v in preds.items()}}, title=f"{dataset} test")
viz.savefig(fig, run_dir, "roc_pr")
plt.show()
pd.Series(list(shares.values())[0]["top_features"]).head(15).to_frame("abs_contribution")

In [ ]:
for name, (pv, pt) in preds.items():
    results.save_predictions(run_dir, name, {"val": (va, y_va, pv), "test": (te, y_te, pt)})
results.save_metrics(run_dir, {"models": rows, "compare": compare, "compare_context": compare_extra,
                               "attribution": {k: {kk: vv for kk, vv in v.items() if kk != "top_features"} for k, v in shares.items()},
                               "params": {"graph_builder": graph_builder, "seed": seed, "quick": quick, "archs": archs, "timings_s": timings}})
summary = {"test_auc": {m: round(r["test"]["auc"], 4) for m, r in rows.items()}, "elapsed_s": round(time.time() - t0, 1)}
ckpt.mark_done(run_dir, summary)
summary